# Patent IPC Classification: Qwen 3 Fine-Tuning Pipeline

This notebook implements an instruction fine-tuning pipeline for the Qwen 3-0.6B model targeting International Patent Classification (IPC). It leverages Low-Rank Adaptation (LoRA) for parameter-efficient fine-tuning and applies Target-Only Masking via cross-entropy loss isolation on assistant responses to ensure optimal gradient stability.

## 1. Colab Environment Setup

Install notebook dependencies, require a GPU runtime in Colab, and set deterministic seeds for reproducibility.

In [ ]:
import sys
import os
import subprocess
import random
import logging
import warnings

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "accelerate>=1.1.0",
        "datasets",
        "matplotlib",
        "peft",
        "scikit-learn",
        "transformers",
    ])

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Colab: Runtime > Change runtime type > GPU')

if torch.cuda.is_available():
    logging.info(f'GPU detected: {torch.cuda.get_device_name(0)}')

class CompletionOnlyDataCollator:
    def __init__(self, tokenizer, response_template, padding=True):
        self.tokenizer = tokenizer
        self.response_template = response_template
        self.response_template_ids = tokenizer.encode(response_template, add_special_tokens=False)
        self.padding = padding

    @staticmethod
    def _find_subsequence(sequence, subsequence):
        if not subsequence:
            return -1

        limit = len(sequence) - len(subsequence) + 1
        for index in range(limit):
            if sequence[index:index + len(subsequence)] == subsequence:
                return index
        return -1

    def __call__(self, features):
        batch = self.tokenizer.pad(features, padding=self.padding, return_tensors="pt")
        labels = batch["input_ids"].clone()

        for row_index, input_ids in enumerate(batch["input_ids"]):
            response_start = self._find_subsequence(input_ids.tolist(), self.response_template_ids)
            if response_start == -1:
                labels[row_index] = -100
            else:
                labels[row_index, :response_start + len(self.response_template_ids)] = -100

        batch["labels"] = labels
        return batch


def set_seed(seed=17):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(17)

## 2. Dataset Initialization

Load the raw patent application dataset, standardize the IPC target variables (truncated to 3 characters), and perform the train-test split. The loader accepts Colab-friendly fallback paths.

In [ ]:
def load_and_preprocess_data(filepath="/content/sample_data/patent_sample_data.csv"):
    candidate_paths = [filepath, "/content/patent_sample_data.csv", "/content/drive/MyDrive/masked-finetuning-qwen/data/patent_sample_data.csv"]
    resolved_path = next((path for path in candidate_paths if os.path.exists(path)), None)
    if resolved_path is None:
        raise FileNotFoundError(f"Could not find patent_sample_data.csv. Checked: {candidate_paths}")

    logging.info(f"Loading dataset from: {resolved_path}")
    df = pd.read_csv(resolved_path)

    def truncate_ipc_codes(ipc_str):
        codes = [c.strip() for c in str(ipc_str).split(",") if c.strip()]
        truncated = [c[:3] for c in codes]
        return ", ".join(list(dict.fromkeys(truncated)))

    df["ipc_codes"] = df["ipc_codes"].apply(truncate_ipc_codes)

    raw_dataset = Dataset.from_pandas(df)
    raw_dataset = raw_dataset.shuffle(seed=42)
    split = raw_dataset.train_test_split(test_size=0.05, seed=42)


    logging.info(f"Train split size: {len(split['train'])}")
    logging.info(f"Eval split size: {len(split['test'])}")

    return split["train"], split["test"]

train_ds, eval_ds = load_and_preprocess_data()

## 3. Model & Tokenizer Initialization

Load the Qwen3-0.6B causal language model in a Colab-friendly way. The model is moved to the active device, gradient checkpointing is enabled, and the tokenizer uses the EOS token as padding.

In [ ]:
model_name = "Qwen/Qwen3-0.6B"
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.info(f"Initializing Tokenizer and Model: {model_name} on {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = model.to(device)

logging.info(f"Model parameters: {model.num_parameters():,}")

## 4. Baseline Evaluation

Establish a zero-shot performance baseline before parameter updates. The chat template enforces role boundaries without activating the internal `<think>` tokens during the evaluation phase.

In [ ]:
def generate_prediction(text, target_model, target_tokenizer):
    messages = [
        {
            "role": "system",
            "content": "You are an expert patent classifier. Classify patents using 3-character IPC codes (e.g. B23, H01). Provide only the codes separated by commas, no duplicates."
        },
        {
            "role": "user",
            "content": f"Patent: {text[:200]}"
        }
    ]

    formatted_text = target_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = target_tokenizer([formatted_text], return_tensors="pt", max_length=512, truncation=True).to(target_model.device)

    with torch.inference_mode():
        generated_ids = target_model.generate(
            **model_inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=True,
            pad_token_id=target_tokenizer.eos_token_id,
            eos_token_id=target_tokenizer.eos_token_id,
        )

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    return target_tokenizer.decode(output_ids, skip_special_tokens=True).strip()

def evaluate_model_predictions(eval_model, model_name, dataset, eval_tokenizer, num_samples=None):
    num_samples = len(dataset) if num_samples is None else num_samples
    predictions, true_labels = [], []

    logging.info(f"Evaluating {model_name} on {num_samples} samples...")

    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        prediction = generate_prediction(sample['text'], eval_model, eval_tokenizer)

        predictions.append(prediction)
        true_labels.append(sample['ipc_codes'])

    return predictions, true_labels

results = {}
logging.info("Running Baseline Evaluation...")
base_preds, base_true_labels = evaluate_model_predictions(model, "Base Qwen 3", eval_ds, tokenizer)
results["Base Qwen 3"] = base_preds

## 5. Dataset Pipeline with Target-Only Masking

We structure the conversational data using strict standard roles (`system`, `user`, `assistant`).
To prevent the model from optimizing cross-entropy loss over the system instructions and user inputs, we use a completion-only collator that masks all tokens preceding the assistant response marker, isolating the loss computation entirely to the model's generated response.

In [ ]:
def format_chat_template(example):
    messages = [
        {
            "role": "system",
            "content": "You are an expert patent classifier. Classify patents using 3-character IPC codes (e.g. B23, H01). Provide only the codes separated by commas, no duplicates."
        },
        {
            "role": "user",
            "content": f"Patent: {example['text'][:200]}"
        },
        {
            "role": "assistant",
            "content": example['ipc_codes']
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )
    return {"formatted_text": formatted_text}

def tokenize_fn(examples):
    return tokenizer(
        examples["formatted_text"],
        truncation=True,
        max_length=512
    )

logging.info("Applying chat templates...")
train_ds_formatted = train_ds.map(format_chat_template)
eval_ds_formatted = eval_ds.map(format_chat_template)

logging.info("Tokenizing datasets...")
tokenized_train_ds = train_ds_formatted.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names + ["formatted_text"])
tokenized_eval_ds = eval_ds_formatted.map(tokenize_fn, batched=True, remove_columns=eval_ds.column_names + ["formatted_text"])

# Configure completion-only masking for assistant responses
response_template = "<|im_start|>assistant\n"
data_collator = CompletionOnlyDataCollator(
    tokenizer=tokenizer,
    response_template=response_template,
)

logging.info(f"Data processing complete. Collator configured with response template: {repr(response_template)}")

## 6. LoRA Configuration

Configure Low-Rank Adaptation (LoRA) to target key projection matrices within the attention mechanism and the MLP blocks. This provides parameter efficiency while maintaining sufficient capacity for domain adaptation.

In [ ]:
import subprocess
import sys

# Upgrade torchao, accelerate, and transformers to compatible versions
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "torchao",
    "accelerate",
    "transformers",
])

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

inst_model = get_peft_model(model, lora_config)
inst_model.print_trainable_parameters()

## 7. Training Loop Execution

Initialize the Trainer with our dataset, configuration, and custom completion-only collator. The settings are reduced to fit typical Colab GPU memory.

In [ ]:
training_args = TrainingArguments(
    output_dir="./instruction_results",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=20,
    learning_rate=1e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=1.0,
    dataloader_drop_last=True,
)

trainer = Trainer(
    model=inst_model,
    args=training_args,
    train_dataset=tokenized_train_ds,
    eval_dataset=tokenized_eval_ds,
    data_collator=data_collator,
)

logging.info("Initiating Training Phase...")
training_output = trainer.train()

logging.info(f"Training completed. Final loss: {training_output.training_loss:.4f}")

output_model_dir = "./qwen3_ipc_finetuned"
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
logging.info(f"Model artifacts saved to {output_model_dir}")

## 8. Memory Management

Force garbage collection and clear CUDA caches prior to executing validation inference.

In [ ]:
import gc

del trainer
del tokenized_train_ds, tokenized_eval_ds
del training_output

gc.collect()
torch.cuda.empty_cache()


logging.info("VRAM structures cleared.")

## 9. Evaluation & Metrics Calculation

Compare the zero-shot baseline against the fine-tuned LoRA checkpoint using Exact Match, Partial Match, Precision, Recall, and F1 metrics.

In [ ]:
logging.info("Running Fine-Tuned Evaluation...")
finetuned_preds, true_labels = evaluate_model_predictions(
    inst_model, "Fine-tuned Qwen 3", eval_ds, tokenizer
)
results["Fine-tuned Qwen 3"] = finetuned_preds

def calculate_exact_match_accuracy(predictions, labels):
    matches = sum(1 for p, t in zip(predictions, labels) if p.strip().lower() == t.strip().lower())
    return matches / len(predictions)

def calculate_partial_match_accuracy(predictions, labels):
    matches = 0
    for pred, true in zip(predictions, labels):
        pred_codes = {c.strip() for c in pred.split(',') if c.strip()}
        true_codes = {c.strip() for c in true.split(',') if c.strip()}
        if pred_codes.intersection(true_codes):
            matches += 1
    return matches / len(predictions)

def calculate_precision_recall(predictions, labels):
    total_precision, total_recall, valid_samples = 0, 0, 0

    for pred, true in zip(predictions, labels):
        pred_codes = {c.strip() for c in pred.split(',') if c.strip()}
        true_codes = {c.strip() for c in true.split(',') if c.strip()}

        if pred_codes and true_codes:
            intersection = pred_codes.intersection(true_codes)
            total_precision += len(intersection) / len(pred_codes)
            total_recall += len(intersection) / len(true_codes)
            valid_samples += 1

    if valid_samples == 0:
        return 0, 0, 0

    avg_precision = total_precision / valid_samples
    avg_recall = total_recall / valid_samples
    f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0

    return avg_precision, avg_recall, f1

metrics_summary = {}
for m_name, preds in results.items():
    exact_acc = calculate_exact_match_accuracy(preds, true_labels)
    partial_acc = calculate_partial_match_accuracy(preds, true_labels)
    precision, recall, f1 = calculate_precision_recall(preds, true_labels)

    metrics_summary[m_name] = {
        'exact_acc': exact_acc, 'partial_acc': partial_acc,
        'precision': precision, 'recall': recall, 'f1': f1
    }

    print(f"\n{m_name} Metrics:")
    print(f"  Exact Match Acc:   {exact_acc:.3f}")
    print(f"  Partial Match Acc: {partial_acc:.3f}")
    print(f"  Avg Precision:     {precision:.3f}")
    print(f"  Avg Recall:        {recall:.3f}")
    print(f"  F1 Score:          {f1:.3f}")

# Plotting metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
model_names = list(results.keys())

exact_accuracies = [metrics_summary[n]['exact_acc'] for n in model_names]
partial_accuracies = [metrics_summary[n]['partial_acc'] for n in model_names]
precisions = [metrics_summary[n]['precision'] for n in model_names]
recalls = [metrics_summary[n]['recall'] for n in model_names]
f1_scores = [metrics_summary[n]['f1'] for n in model_names]

# Bar charts Configuration
axes[0, 0].bar(model_names, exact_accuracies, alpha=0.8, label='Exact Match', color='#2b8cbe')
axes[0, 0].bar(model_names, partial_accuracies, alpha=0.5, label='Partial Match', color='#a6bddb')
axes[0, 0].set_title('Accuracy Comparison')
axes[0, 0].set_ylabel('Score')
axes[0, 0].legend()

x = np.arange(len(model_names))
width = 0.35
axes[0, 1].bar(x - width/2, precisions, width, label='Precision', alpha=0.8, color='#e6550d')
axes[0, 1].bar(x + width/2, recalls, width, label='Recall', alpha=0.8, color='#fdae6b')
axes[0, 1].set_title('Precision vs Recall')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(model_names)
axes[0, 1].legend()

axes[1, 0].bar(model_names, f1_scores, alpha=0.8, color='#31a354')
axes[1, 0].set_title('F1 Score Comparison')
axes[1, 0].set_ylabel('F1 Score')

metrics = ['Exact Acc', 'Partial Acc', 'Precision', 'Recall', 'F1']
base_scores = [exact_accuracies[0], partial_accuracies[0], precisions[0], recalls[0], f1_scores[0]]
ft_scores = [exact_accuracies[1], partial_accuracies[1], precisions[1], recalls[1], f1_scores[1]]

axes[1, 1].plot(metrics, base_scores, 'o-', label=model_names[0], linewidth=2, color='#756bb1')
axes[1, 1].plot(metrics, ft_scores, 's-', label=model_names[1], linewidth=2, color='#3182bd')
axes[1, 1].set_title('Aggregated Performance Trajectory')
axes[1, 1].set_ylabel('Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()